# Preprocess CAMS data for dust tool

Interpolate CAMS data to EEA station locations, account for the local timezone, and average per day so that the resulting netcdf file is smaller

## Importing modules

In [ ]:
import pandas as pd
import os
import glob
import xarray as xr

## Loading data

In [ ]:
# Load cams dust data
netcdf_files = glob.glob(os.path.join('CAMS_data', '*dust*.nc'))
datasets = [xr.open_dataset(file) for file in netcdf_files]
cams = xr.concat(datasets, dim='time')
cams = cams.sortby('time')

In [ ]:
#Load observation data
folder_path = './EEA_data'
parquet_files = glob.glob(os.path.join(folder_path, "*.parquet"))
dataframes = [pd.read_parquet(file) for file in parquet_files]
obs = pd.concat(dataframes, ignore_index=True)

# Set Start as datetime
obs['Start'] = pd.to_datetime(obs['Start'])

# Drop unnecessary columns
obs = obs.drop(columns=['End', 'ResultTime', 'Pollutant',
       'Unit', 'AggType', 'DataCapture','FkObservationLog'])

# Rename 'Value' column
obs = obs.rename(columns={'Value': 'observed_PM10'})
obs['observed_PM10'] = obs['observed_PM10'].astype(float)

# Flag exceedance days with concentration above 50 µg m-3
obs['Exceedance'] = obs['observed_PM10'] > 50

## Filter observation data

In [ ]:
#Count and filter observation data
print(f"Number of observations before filtering: {len(obs)}")
obs = obs[obs['Validity'] > 0]
print(f"Number of observations after validity filter: {len(obs)}")
obs = obs[obs['Verification'] == 1]
print(f"Number of observations after verification filter: {len(obs)}")

In [ ]:
# Filter for stations with 2024 data
stations_with_2024_data = obs[
    (obs['Start'] >= '2024-01-01') & (obs['Start'] < '2025-01-01')
]['Samplingpoint'].unique()
obs = obs[obs['Samplingpoint'].isin(stations_with_2024_data)]
print(f"Number of observations after filtering for 2024 stations: {len(obs)}")

## Add metadata to obervation data for timezone, longitude and latitude

In [ ]:
#Load metadata observational data
metadata = pd.read_csv('DataExtract.csv', low_memory = False)

# Retrieve stations from observational data
all_stations = pd.DataFrame({'Samplingpoint': obs['Samplingpoint'].unique()})

# Create dataframe with all stations and their location
metadata['Samplingpoint'] = metadata['Air Quality Station EoI Code'].str[:2] + '/' + metadata['Sampling Point Id']
stations = all_stations.merge(
    metadata[['Samplingpoint', 'Longitude', 'Latitude', 'Altitude','Timezone']], 
    on='Samplingpoint', 
    how='left'
).drop_duplicates(subset=['Samplingpoint'], keep='first')

# Add station coordinates to obs
obs = obs.merge(stations[['Samplingpoint', 'Latitude', 'Longitude','Altitude','Timezone']], 
                on='Samplingpoint', 
                how='left')

## Preprocess CAMS data

In [ ]:
# Definition to extract timezone
def extract_timezone_offset(tz_string):
    if tz_string == 'UTC':
        return 0
    else:
        return int(tz_string.replace('UTC+', ''))

# Definition to interpolate from closest model gridpoints
def interpolate_local_region(ds, lat, lon, buffer=0.25):
    local_region = ds.sel(
        lat=slice(lat-buffer, lat+buffer),
        lon=slice(lon-buffer, lon+buffer)
    )
    return local_region.interp(lat=lat, lon=lon, method='linear')

# Flag dust for each samplingpoint separately
unique_stations = obs['Samplingpoint'].unique()

datasets = []

for i, station in enumerate(unique_stations):
    print(f"\rProcessing station {i+1}/{len(unique_stations)}", end="")
    
    # Get station coordinates and timezone
    station_obs = obs[obs['Samplingpoint'] == station]
    lat = station_obs['Latitude'].iloc[0]
    lon = station_obs['Longitude'].iloc[0]
    tz_offset = extract_timezone_offset(station_obs['Timezone'].iloc[0])
    
    # Interpolate CAMS data to this station location
    station_cams = interpolate_local_region(cams, lat, lon)
    
    # Convert UTC to local timezone
    station_cams['time_local'] = station_cams['time'] + pd.to_timedelta(tz_offset, unit='h')
    
    # Calculate daily average dust using local timezone
    daily_cams = station_cams.groupby(station_cams['time_local'].dt.date).mean()
    datasets.append(daily_cams)

In [ ]:
daily_cams = xr.concat(datasets, dim='station')
daily_cams = daily_cams.assign_coords(station=unique_stations)

## Write netcdf file

In [ ]:
if 'date' in daily_cams.coords:
    daily_cams = daily_cams.assign_coords(date=pd.to_datetime(daily_cams['date'].values))

daily_cams.to_netcdf('daily_cams.nc')